[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uahccre/ncs_workshop/blob/main/agentic_workshop.ipynb)

## Making a Copy of the Notebook
1. Navigate to the [UAH CCRE GitHub](https://github.com/uahccre/ncs_workshop/blob/main/agentic_workshop.ipynb).
2. Click the "Open in Colab" Button
3. The file will open in Read Only mode. Go to File -> Save a Copy in Drive
4. A new tab will open with your editable copy for the rest of the lab.
5. Go to "Edit" and select "Clear All Outputs" to ensure we're starting fresh.

## Setting up the Anthropic API Key
1. Visit Bitwarden Send (See slides)
2. Enter the password provided by the instructor
3. Copy the API key value
4. Click the "key" on the left of the screen to open the secrets panel
5. Click "Add New Secret"
6. Name it exactly: `ANTHROPIC_API_KEY`
7. Value: paste the API key value you just copied from Bitwarden Send
8. Turn on the "Notebook access" toggle

## Install Required Packages
We'll install `smolagents`, `markdownify`, and `wikipedia-api`. These packages are required for the notebook and don't ship with Colab.

In [ ]:
!pip install -q "smolagents[litellm]"

In [ ]:
# VisitWebpageTool needs markdownify
!pip install -q markdownify

In [ ]:
# Fallback if DuckDuckGo fails
!pip install -q wikipedia-api

## Loading Claude Haiku
SmolAgents uses underlying LLMs to do orchestration and reasoning. We'll use the lightweight Claude Haiku model for this workshop.

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

from smolagents import LiteLLMModel
model = LiteLLMModel(model_id="anthropic/claude-haiku-4-5-20251001")
print("Model loaded from Anthropic")

## Our First Agent
An **agent** is just three things: a language model, a set of tools, and a loop that lets it act, observe the result, and act again until the task is done.

This act -> observe -> repeat pattern has a name you'll see all over the field: **ReAct** (Reason + Act). SmolAgents puts its own spin on it. With a **CodeAgent**, the agent's "action" is actual **Python code** it writes and runs, rather than plain text instructions. It writes code, runs it, sees the output, and continues. It's a very natural way to compute, chain steps, and use tools.

We'll start with no tools at all. A CodeAgent can still write and run Python and we can watch the reasoning loop before we add anything else.

### Why Not Just Ask the Model Directly?
A plain language model call is one shot. You send a prompt, you get back text, and that's it. It can't look anything up, run a calculation, or check its own work after the initial response. So if the answer needs a live fact or a real computation, a lone model just guesses.



An agent wraps the same model in a loop that lets it actually act:
$$\text{Think} \rightarrow \text{Act} \rightarrow \text{Observe} \rightarrow \text{Repeat until done} \rightarrow \text{Final Answer}$$


Each time around the loop, the model sees what happend on the previous step and decides what to do next. Search again, fix its generated code, or finish. That "act -> observe -> adjust" cycle is what turns a text generation model into something that can complete tasks.

> One important caveat. Frontier models are actually agentic now. Claude, ChatGPT, and Gemini all do these things. But any locally hosted model like Llama cannot without this agentic loop structure.

In [ ]:
from smolagents import CodeAgent

# tools=[] (empty array) is purposeful below. The agent still has a built-in
# Python interpreter without specifying it. verbosity_level=1 allows us to watch
# it think and reason -> write code -> observe -> repeat.
agent = CodeAgent(tools=[], model=model, max_steps=5, verbosity_level=1)

result = agent.run("What is the sum of all prime numbers below 50?")

print("\nFinal answer:\n", result)

The output above the "Final answer:" is the agent loop itself. We can see the function it defined and the steps it took to come up with the sum before execution completed.

## Giving our Agent a Tool
Out of the box, our agent can only reason and compute. **Tools** let it *do* things like look something up, call an Application Programming Interface (API), run a calculation you define, etc.

In SmolAgents, a tool is just a Python function with two things added:
- the `@tool` decorator
- a clear **docstring** and **type hints**

That documentation isn't just for show. The model reads it to decide when to call your tool and what to pass in to the call. A vague docstring gives you a confused agent, so this is where careful writing is critical.

We'll build a tool that fetches the current weather for any city.

> *Important to note*: If you happen to get a `docstring` error when you execute the below code block, it likely is caused by a trailing space after the `Args:` or a missing argument description.

In [ ]:
import requests
from smolagents import tool

@tool
def get_weather(city: str) -> str:
  """
    Gets the current weather for a given city.

    Args:
      city: The name of the city
  """
  # Turn the city name into coordinates
  geo = requests.get(
      "https://geocoding-api.open-meteo.com/v1/search",
      params={"name": city, "count": 1},
      timeout=20,
  ).json()

  if not geo.get("results"):
    return f"Could not finda  location named '{city}'."

  place = geo["results"][0]
  lat, lon = place["latitude"], place["longitude"]
  label = f"{place['name']}, {place.get('country', '')}".strip(", ")

  # Look up current conditions at those coords
  forecast = requests.get(
      "https://api.open-meteo.com/v1/forecast",
      params={
          "latitude": lat,
          "longitude": lon,
          "current": "temperature_2m,wind_speed_10m,weather_code"
      },
      timeout=20
  ).json()

  current = forecast["current"]
  temp, wind, code = (
      current["temperature_2m"],
      current["wind_speed_10m"],
      current["weather_code"],
  )

  # A small slice of weather codes -> plain English
  conditions = {
        0: "clear sky", 1: "mostly clear", 2: "partly cloudy", 3: "overcast",
        45: "foggy", 48: "rime fog",
        51: "light drizzle", 53: "drizzle", 55: "heavy drizzle",
        61: "light rain", 63: "rain", 65: "heavy rain",
        71: "light snow", 73: "snow", 75: "heavy snow",
        80: "rain showers", 81: "rain showers", 82: "violent rain showers",
        95: "thunderstorm",
  }

  sky = conditions.get(code, f"weather code {code}")

  return f"Current weather in {label}: {sky}, {temp}\u00b0C, wind {wind} km/h."


Take a look at our function definiton in the above cell.

The `@tool` decorator is used to tell the agent this is a tool it's allowed to use.

The actual definition `def get_weather(city: str) -> str:` contains two type hints. `city: str` tells the agent that a string must be passed into this function, where ` -> str:` tells the agent that a string will be returned from this function.

Lastly, everything between the `""" ... """` triple quotes is our docstring. You can see our docstring describes what this tool does as well as what arguments it accepts and a description of what the arguments should contain.

Now let's test our tool.

In [ ]:
print(get_weather(city="Huntsville, Alabama"))

## Letting the Agent Use the Tool
Now we can give the tool to our CodeAgent by passing it in in the `tools` list. Watch what happens when we ask a question that *requires* the tool and then a bit of judgment on top of the result. The agent has to decide to call `get_weather`, read what comes back, and reason about it.

In [ ]:
from smolagents import CodeAgent

agent = CodeAgent(tools=[get_weather], model=model, max_steps=5, verbosity_level=1)

result = agent.run("Should I bring a jacket to work in Huntsville, Alabama today?")

print("\nFinal Answer:\n", result)

## An Agent that Researches the Web
So far our agent only uses one tool we wrote. SmolAgents also ships with **built-in tools** including web search and a "read this webpage" tool.

Now we'll give the agent two tools at once and a real reearch task. Watch how it chains them: it decides what to search, reads the results, optionally opens a page for detail, and synthesizes an answer. Deciding *which* tool to use *when* is the heart of what makes an agent an agent.

In [ ]:
from smolagents import CodeAgent, WebSearchTool, VisitWebpageTool

research_agent = CodeAgent(
    tools=[WebSearchTool(max_results=5), VisitWebpageTool()],
    model=model,
    max_steps=6,
    verbosity_level=1
)
print("Research agent is ready")

In [ ]:
result = research_agent.run(
    "Search the web for the smolagents library and tell me, in 3 sentences, "
    "what it is and what makes it different from other agent frameworks."
)

print("\nFinal answer:\n", result)

>**Possible Failure Mode**: DuckDuckGoSearch has been known to rate limit searches coming from the same set of IP addresses. Since we're all in the same room, this is likely to happen. If everyone starts getting rate limited, we can swap to the Wikipedia Search below instead

In [ ]:
from smolagents import CodeAgent, WikipediaSearchTool, VisitWebpageTool

research_agent = CodeAgent(
    tools=[WikipediaSearchTool(), VisitWebpageTool()],
    model=model,
    max_steps=6,
    verbosity_level=1,
)

result = research_agent.run(
    "Look up the city of Huntsville, Alabama and tell me in 3 sentences "
    "why it's historically significant."
)

print("\nFinal Answer:\n", result)

## Building a Multi-Agent System
Everything so far has just been one agent. Real systems most often use several agents. One is a manager that coordinates specialist agents with one role.

We'll build a multi-agent system reusing what we've already made:
- A weather specialist that owns the `get_weather` tool
- A research specialist that owns the `web search` and `read webpage` tools
- A manager that owns no tools but delegates and combines answers

$$\text{Manager} \rightarrow \text{delegates to specialists} \rightarrow \text{combines their answers} \rightarrow \text{Final Answer}$$

The manager decides *who* handles each part of a request, hands off the sub-task, and weaves the results together. And it's loops all the way down: each specialist runs its own Think → Act → Observe loop, and the manager runs one too — agents calling agents.

In [ ]:
from smolagents import CodeAgent, WebSearchTool, VisitWebpageTool

# Specialist 1: reuses the weather tool we wrote earlier
weather_agent = CodeAgent(
    tools=[get_weather],
    model=model,
    name="weather_agent",
    description="Looks up the current weather for a given city. Give it a city name.",
    max_steps=4
)

# Specialist 2: reuses the web tools from the last section
research_agent = CodeAgent(
    tools=[WebSearchTool(max_results=5), VisitWebpageTool()],
    model=model,
    name="research_agent",
    description="Searches the web and reads pages to answer factual questions "
                "about places, events, or topics",
    max_steps=6,
)

# The manager: no tools of its own, just coordinates the two specialists
manager = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[weather_agent, research_agent],
    max_steps=6,
    verbosity_level=1,
)

print("Multi-agent system ready: manager + weather agent + research agent")

In [ ]:
result = manager.run(
    "I'm visiting Huntsville, Alabama this weekend. Check the current weather there, "
    "and find one outdoor attraction worth visiting. Then tell me whether it's a "
    "good weekend to go, and why."
)

print("\nFinal answer:\n", result)

> **Fallback in case we all get rate limited**

In [ ]:
from smolagents import CodeAgent, WikipediaSearchTool, VisitWebpageTool

# Specialist 1: reuses the weather tool we wrote earlier
weather_agent = CodeAgent(
    tools=[get_weather],
    model=model,
    name="weather_agent",
    description="Looks up the current weather for a given city. Give it a city name.",
    max_steps=4
)

# Specialist 2: reuses the web tools from the last section
research_agent = CodeAgent(
    tools=[WikipediaSearchTool(), VisitWebpageTool()],
    model=model,
    name="research_agent",
    description="Looks up factual information about places, events or topics "
                "using Wikipedia and can read webpages.",
    max_steps=6,
)

# The manager: no tools of its own, just coordinates the two specialists
manager = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[weather_agent, research_agent],
    max_steps=6,
    verbosity_level=1,
)

print("Multi-agent system ready: manager + weather agent + research agent")

In [ ]:
result_fallback = manager.run(
    "I'm visiting Huntsville, Alabama this weekend. Check the current weather there, "
    "and find one outdoor attraction worth visiting. Then tell me whether it's a "
    "good weekend to go, and why."
)

print("\nFinal answer:\n", result_fallback)

### Reading the Trace: Agents Calling Agents

The output above is busier than the earlier runs. That's the multi-agent system working. Worth pointing out as you scroll through it:

- The **manager** starts, and instead of calling a tool directly, it **delegates**: you'll see it hand a sub-task to `weather_agent` and to `research_agent`.
- Each specialist then runs its **own** Think → Act → Observe loop, nested inside the manager's run. The weather one calling `get_weather`, the research one searching and reading a page.
- Control returns to the **manager**, which takes both specialists' answers and combines them into the single recommendation at the very bottom.

That nesting (one agent's loop running other agents' loops) is the whole idea of multi-agent orchestration. Notice nobody scripted "first check weather, then search": the manager decided how to break up the task on its own.

## Before You Turn Agents Loose: Limits and Safety

Agents are powerful because they can act on their own. Which is exactly why they need guardrails. A few things to keep in mind before building on this:

- **They can loop or run away.** An agent that never decides it's "done" will keep going, burning time and money. That's why every agent here sets `max_steps`, a hard cap on how many times it can loop.
- **Cost adds up.** Every step is a model call you pay for, and multi-agent runs multiply those calls fast. Watch usage when a manager is coordinating several specialists.
- **Running generated code is risky.** A `CodeAgent` executes code the model writes. smolagents sandboxes this and only allows a safe list of imports. But if you loosen that, you're running unreviewed code on your machine. Be deliberate about what you authorize.
- **Tool access is real access.** An agent with a tool that sends email, edits files, or hits an API can actually *do* those things, including the wrong ones. Give agents the least access they need, and keep destructive actions behind human approval.
- **They can be fooled.** A web page or document an agent reads can contain instructions that try to hijack it (*prompt injection*). Treat anything an agent pulls from the outside world as untrusted input, not commands.
- **Confident does not mean correct.** Like any model, an agent can be wrong while sounding sure. Keep a human in the loop for anything that matters.

An agent is only as safe as the tools you give it and the limits you put around it. Start small, cap the loops, sandbox the code, and keep a person in the decision loop for anything consequential.